In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import os

In [12]:
full_newsgroups = fetch_20newsgroups(subset='all', shuffle=True, random_state=42)
all_target_names = full_newsgroups.target_names

print("All available categories in the 20 Newsgroups dataset:")
for category_name in all_target_names:
    print(f"- {category_name}")

All available categories in the 20 Newsgroups dataset:
- alt.atheism
- comp.graphics
- comp.os.ms-windows.misc
- comp.sys.ibm.pc.hardware
- comp.sys.mac.hardware
- comp.windows.x
- misc.forsale
- rec.autos
- rec.motorcycles
- rec.sport.baseball
- rec.sport.hockey
- sci.crypt
- sci.electronics
- sci.med
- sci.space
- soc.religion.christian
- talk.politics.guns
- talk.politics.mideast
- talk.politics.misc
- talk.religion.misc


In [13]:
categories = [
    'comp.graphics',
    'comp.os.ms-windows.misc',
    'comp.sys.ibm.pc.hardware',
    'comp.sys.mac.hardware',
    'rec.autos',
    'rec.motorcycles',
    'rec.sport.baseball',
    'rec.sport.hockey',
    'sci.crypt',
    'sci.electronics'
]

newsgroups_data = fetch_20newsgroups(subset='all', categories=categories,
                                     shuffle=True, random_state=42,
                                     remove=('headers', 'footers', 'quotes'))
docs = newsgroups_data.data
target_names = newsgroups_data.target_names

print(f"Number of documents: {len(docs)}")
print(f"Number of categories: {len(target_names)}")

Number of documents: 9857
Number of categories: 10


In [14]:
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.85, # Ignore terms that appear in more than 85% of the documents
    min_df=2,    # Ignore terms that appear in less than 2 documents
    stop_words='english', # Remove common English stop words
    norm='l2' # Apply L2 normalization
)

tfidf_matrix = tfidf_vectorizer.fit_transform(docs)

print(f"Shape of TF-IDF matrix: {tfidf_matrix.shape}")

feature_names = tfidf_vectorizer.get_feature_names_out()

Shape of TF-IDF matrix: (9857, 31617)


In [15]:
n_components_settings = [50, 100, 200] # Stated in problem statement
svd_results = []

for n_components in n_components_settings:
    print(f"\n{n_components} components SVD")
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    lsa_matrix = svd.fit_transform(tfidf_matrix)

    explained_variance_ratio = svd.explained_variance_ratio_.sum()
    print(f"  Total explained variance ratio: {explained_variance_ratio:.4f}")

    svd_results.append({
        'n_components': n_components,
        'explained_variance_ratio': explained_variance_ratio,
        'svd_model': svd,
        'lsa_matrix': lsa_matrix
    })


50 components SVD
  Total explained variance ratio: 0.0762

100 components SVD
  Total explained variance ratio: 0.1206

200 components SVD
  Total explained variance ratio: 0.1887


In [16]:
final_results = []

for result in svd_results:
    n_components = result['n_components']
    lsa_matrix = result['lsa_matrix']
    explained_variance_ratio = result['explained_variance_ratio']

    n_clusters = len(target_names)
    print(f"\nKMeans clustering with {n_clusters} clusters for {n_components} components")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(lsa_matrix)

    # Silhouette score is only valid if n_samples > 1 and 1 < n_clusters < n_samples - 1
    if n_clusters < len(lsa_matrix) - 1 and len(lsa_matrix) > 1:
        silhouette_avg = silhouette_score(lsa_matrix, kmeans_labels)
        print(f"  Silhouette Score: {silhouette_avg:.4f}")
    else:
        silhouette_avg = None
        print("  Cannot compute Silhouette Score (insufficient data or clusters).")

    final_results.append({
        'n_components': n_components,
        'explained_variance_ratio': explained_variance_ratio,
        'silhouette_score': silhouette_avg,
        'svd_model': result['svd_model']
    })

results_df = pd.DataFrame([{
    'components': r['n_components'],
    'metric_value': r['silhouette_score']
} for r in final_results])
results_df.to_csv('svd_results.csv', index=False)
print("\nResults saved to svd_results.csv")


KMeans clustering with 10 clusters for 50 components
  Silhouette Score: 0.0934

KMeans clustering with 10 clusters for 100 components
  Silhouette Score: 0.0462

KMeans clustering with 10 clusters for 200 components
  Silhouette Score: 0.0100

Results saved to svd_results.csv


In [19]:
for result in final_results:
    n_components = result['n_components']
    svd_model = result['svd_model']
    components = svd_model.components_

    print(f"\nInterpreting topics for SVD with {n_components} components.")

    output_file_path = f'lsa_topic_terms_{n_components}_components.txt'
    with open(output_file_path, 'w') as f_out:
        f_out.write(f"Top 10 terms for topics (SVD with {n_components} components):\n\n")
        print(f"\nTop 10 terms for topics ({n_components} components):\n")
        num_topics_to_show = min(5, components.shape[0])
        for i, topic in enumerate(components[:num_topics_to_show]):
            top_n_terms_indices = topic.argsort()[:-11:-1]
            top_terms = [feature_names[j] for j in top_n_terms_indices]
            print(f"Topic {i+1}: {', '.join(top_terms)}")
            f_out.write(f"Topic {i+1}: {', '.join(top_terms)}\n")
    print(f"\nTopic terms saved to {output_file_path}")


Interpreting topics for SVD with 50 components.

Top 10 terms for topics (50 components):

Topic 1: know, like, just, don, windows, does, use, thanks, think, drive
Topic 2: windows, card, dos, thanks, drive, file, scsi, files, program, disk
Topic 3: key, chip, encryption, clipper, government, keys, escrow, algorithm, law, phone
Topic 4: drive, scsi, ide, drives, controller, hard, bus, disk, floppy, hd
Topic 5: windows, dos, scsi, game, key, drive, file, disk, team, os

Topic terms saved to lsa_topic_terms_50_components.txt

Interpreting topics for SVD with 100 components.

Top 10 terms for topics (100 components):

Topic 1: know, like, just, don, windows, does, use, thanks, think, drive
Topic 2: windows, card, dos, thanks, drive, file, scsi, files, program, disk
Topic 3: key, chip, encryption, clipper, government, keys, escrow, algorithm, law, phone
Topic 4: drive, scsi, ide, drives, controller, hard, bus, disk, floppy, hd
Topic 5: windows, dos, scsi, game, key, drive, file, disk, tea

Because Truncated SVD finds and extracts the most important underlying patterns (latent subjects) while discarding noise and less significant information, it is especially useful for sparse text matrices. The number of features is greatly reduced by projecting the high-dimensional sparse data onto a much lower-dimensional space, which makes the data easier to handle and enhances the performance of subsequent tasks like clustering or classification. It basically reveals the text's "hidden meanings" without requiring the enormous number of individual words to be explicitly processed.